In [10]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [11]:
df_dk = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/dk_bn_no_context.csv")

In [12]:
df_dk = df_dk.drop(["Unnamed: 0"], axis = 1)

In [13]:
df_dk

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,319173.149269,22.06,76.414286,6.45,6.42,7.49,17.747,2.747300,24.277939,5.548536,3.69,5.3,11.7,0.000010,304.029304,1.866582
1,329422.223703,22.61,76.414286,6.44,6.23,9.47,18.543,3.471527,24.986449,3.835619,0.62,6.5,11.8,0.000009,297.632058,2.784574
2,313144.896657,19.85,76.100000,6.43,6.06,13.44,19.948,4.602963,27.815694,-2.111442,1.17,4.2,13.1,0.000009,275.000000,6.041840
3,326784.316443,19.37,74.900000,6.37,6.17,15.55,21.888,6.793833,27.336108,-2.441067,-3.57,4.7,13.3,0.000009,292.972973,7.495645
4,331838.652835,19.85,74.800000,6.32,6.29,16.31,23.389,6.154324,26.540559,-1.887012,3.94,3.6,12.1,0.000008,282.585278,16.993114
5,338534.135302,19.85,74.300000,5.97,5.73,15.77,25.465,5.799405,26.500323,-3.257951,-5.00,4.0,12.0,0.000007,251.520572,6.354829
6,344715.698783,19.73,74.300000,5.64,5.45,14.77,27.173,6.229005,25.928504,-1.181728,0.20,4.4,11.9,0.000007,239.037433,8.319061
7,350893.947080,19.40,74.700000,5.56,5.43,14.20,29.310,6.455948,25.797420,1.252108,1.86,5.8,12.1,0.000006,244.148936,23.835000
8,357211.575191,19.56,75.400000,5.58,5.59,12.15,30.469,6.498281,25.546356,-1.143677,0.61,8.4,12.2,0.000006,257.394366,12.745287
9,366884.878867,20.23,76.000000,5.80,5.70,12.15,31.715,6.035122,24.942915,-0.070663,2.50,7.2,11.9,0.000006,270.855148,0.401479


In [17]:
import pandas as pd
from causalnex.discretiser import Discretiser

def discretize_dataframe(df, max_buckets=4):
    discretised_df = df.copy()
    
    for col in df.columns:
        unique_vals = df[col].nunique()
        
        # 1. If the column is constant, set to 0
        if unique_vals <= 1:
            discretised_df[col] = 0
            continue
            
        # 2. Adjust buckets if unique values are fewer than max_buckets
        # This prevents the Discretiser from failing on low-variance data
        n_buckets = min(max_buckets, unique_vals)
        
        try:
            # 3. Use CausalNex Discretiser
            d = Discretiser(method="quantile", num_buckets=n_buckets)
            discretised_df[col] = d.fit_transform(df[[col]].values).flatten()
            
        except (ValueError, IndexError):
            # 4. Fallback: If quantiles fail (e.g., too many identical values), 
            # use a simple dense rank to force discretization
            discretised_df[col] = df[col].rank(method='dense').astype(int) - 1
            
    return discretised_df.astype(int)

# Usage:
discretised_dk = discretize_dataframe(df_dk, max_buckets=4)

print("Discretised data head:")
print(discretised_dk.head())

Discretised data head:
   GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  CO2C  \
0     0     3    2    3    3    0     0     0     0     3    3   2    0     3   
1     0     3    2    3    3    0     0     0     2     3    1   3    0     3   
2     0     2    1    3    2    2     0     0     3     0    2   1    3     3   
3     0     0    1    3    3    3     0     3     3     0    0   1    3     3   
4     1     2    0    2    3    3     1     2     3     0    3   0    1     2   

   PAM  EUPC  
0    3     0  
1    3     0  
2    2     0  
3    3     1  
4    3     3  


In [19]:
from causalnex.structure.notears import from_pandas
sm = from_pandas(df_dk)

In [20]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("graphs/dk_fully_connected.html")

graphs/dk_fully_connected.html


In [21]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("graphs/dk_thresholded.html")

graphs/dk_thresholded.html


In [22]:
for cause, effect in sm.edges:
    print(f"{cause} -> {effect}")

VABI -> GDPC
EMP -> GDPC
EMP -> PAM
QOA -> GDPC
QOR -> GDPC
YUA -> GDPC
REST -> GDPC
REST -> PAM
EBSG -> GDPC
GGFC -> GDPC
GGFC -> PAM
LBGD -> GDPC
FDI -> GDPC
PC -> GDPC
ARP -> GDPC
PAM -> GDPC
PAM -> EMP
PAM -> REST
PAM -> GGFC
EUPC -> GDPC
